# 01 — Annual minimum low-flow frequency analysis

Port of `Marginal.R`. Fits GEV / Pearson III / Gamma / Weibull to the **annual** minimum 7-day mean discharge, selects by AIC, tests by bootstrap Anderson-Darling, and reports return levels with confidence intervals.

In [ ]:
# --- 1. Which interpreter is this, and does it have the stack? -------------
# A "ModuleNotFoundError: No module named 'numpy'" means this kernel is not the
# environment you installed into. The pip line printed below targets THIS
# kernel, which is what you want -- a bare `pip install` may hit a different one.
import importlib.util, sys

print("interpreter :", sys.executable)
print("version     :", sys.version.split()[0])

REQUIRED = ["numpy", "scipy", "pandas", "matplotlib", "statsmodels",
            "openpyxl", "lmoments3"]
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]

if missing:
    print("\nMISSING:", ", ".join(missing))
    print("\nInstall into THIS kernel by running the following in a terminal")
    print("(the quotes matter -- the path may contain spaces):\n")
    print(f'  "{sys.executable}" -m pip install ' + " ".join(missing))
    print("\nOr from inside this notebook, in a new cell:\n")
    print("  %pip install " + " ".join(missing))
    print("\nIf you use conda, prefer:")
    print("  conda install -n <env> " + " ".join(m for m in missing if m != "lmoments3"))
    print("  conda run -n <env> pip install lmoments3")
else:
    print("\nAll dependencies present.")

In [ ]:
# --- 2. Locate the lowflow package ----------------------------------------
# Searches this directory and its parents for lowflow/__init__.py, so the
# notebook works whether it sits beside the package or in a notebooks/
# subfolder. Set PKG_ROOT by hand if your layout is unusual.
import os, sys, pathlib

PKG_ROOT = None          # e.g. pathlib.Path("/path/to/lowflow-frequency-analysis")

if PKG_ROOT is None:
    here = pathlib.Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "lowflow" / "__init__.py").exists():
            PKG_ROOT = cand
            break

if PKG_ROOT is None:
    raise FileNotFoundError(
        "Could not find lowflow/__init__.py in this directory or any parent.\n"
        "Set PKG_ROOT above to the folder that CONTAINS the lowflow/ directory, "
        "e.g. pathlib.Path('/path/to/lowflow-frequency-analysis')"
    )

PKG_ROOT = pathlib.Path(PKG_ROOT).resolve()
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
os.chdir(PKG_ROOT)       # so relative data/ and out/ paths resolve

import lowflow
print("lowflow package :", lowflow.__file__)
print("working dir     :", os.getcwd())
print("version         :", lowflow.__version__)

In [ ]:
import numpy as np
import pandas as pd

from lowflow import (read_flow, block_minima, scan_seasons, season_diagnostic,
                     fit_all, compare, trend_tests)
from lowflow.marginals import GEVMinima, GEVMinimaNS, PearsonIII, lr_test
import lowflow.figures as figs

figs.apply_style()
pd.set_option("display.width", 170, "display.max_columns", 40)

## 1. Read and screen the record

`read_flow` regularises to a gap-free daily index **before** computing the D-day mean, so a window spanning a gap is NaN rather than an average over fewer days. `D = 7` is the 7-day mean behind MAM7 / 7Q10.

In [ ]:
# Point these at your own workbook; the two columns must be date and flow.
STATION  = 'Station A'
PATH     = 'data/station_a.xlsx'
SEASON   = None
D        = 7          # 7-day mean (MAM7 / 7Q10). D = 1 is the single-day minimum.
MIN_COV  = 0.90       # a block needs 90% of its days to be used
T        = (2, 5, 10, 20, 50, 100)
B_GOF    = 199        # bootstrap replicates; raise to 999 for publication
B_CI     = 499        # raise to 1999 for publication

rec = read_flow(PATH, name=STATION, D=D, zero_policy="keep")
print(rec.summary())

## 2. Is the season the right one?

`capture_rate` is the fraction of annual minima the window actually contains. Below about 0.8 the seasonal analysis is not describing the annual drought, whatever care goes into the rest of it.

In [ ]:
scan = scan_seasons(rec, min_coverage=MIN_COV)
print(scan.head(10).drop(columns=["months"]).to_string(index=False))

if SEASON is not None:
    from lowflow.blocks import season_label
    lab = season_label(SEASON)
    row = scan.loc[scan.window == lab]
    if len(row):
        cr = float(row.capture_rate.iloc[0])
        print(f"\nchosen window {lab}: capture rate {cr:.1%}")
        if cr < 0.8:
            print(f"WARNING: {lab} contains the annual minimum in only {cr:.0%} of years."
                  f" Best is {scan.window.iloc[0]} at {scan.capture_rate.iloc[0]:.0%}.")
    fig = figs.fig_season_scan(scan, chosen=lab,
                               title=f"{STATION}: does the window contain the drought?")

## 3. Block minima, with coverage screening

Rejected blocks are kept in the table and drawn as open markers, so a year with a gauge outage never contributes a spurious minimum.

In [ ]:
bm = block_minima(rec, season=SEASON, min_coverage=MIN_COV)
print(bm.summary())
x, years = bm.values, bm.years

In [ ]:
tr = trend_tests(x, years)
print(tr.to_string(index=False))
fig = figs.fig_block_minima(bm, tr)

## 4. Candidate distributions

Selection is by **AIC**; the test is a parametric-bootstrap Anderson-Darling. The `KS_p_invalid` column is printed only to show why it must not be used for selection: with parameters estimated in-sample its rejection rate at alpha = 0.05 is 0.000 and its median p-value 0.92.

Watch the `notes` column. A Pearson III whose fitted support bound lands above the sample minimum has **zero density** at an observed value, so its log-likelihood is -inf and its AIC infinite -- while KS may still rank it first.

In [ ]:
fits = fit_all(x)
cmp = compare(fits, B=B_GOF, rng=np.random.default_rng(20240617))
print(cmp.to_string(index=False))

ok = {k: v for k, v in fits.items() if not isinstance(v, Exception)}
best = cmp.loc[0, "distribution"]
print(f"\nselected by AIC: {best}")
for nm, f in ok.items():
    if f.notes:
        print(f"  note ({nm}): {'; '.join(f.notes)}")

In [ ]:
fig = figs.fig_density_fits(x, ok, title=f"{STATION}: fitted densities")

In [ ]:
fig = figs.fig_qq(x, ok, title=f"{STATION}: Q-Q")

In [ ]:
fig = figs.fig_lmom_diagram(x, title=f"{STATION}: L-moment ratio diagram")

## 5. Return levels, with confidence intervals

The T-year low flow is the discharge with non-exceedance probability 1/T. The interval is the point of this section: with ~50 block minima the 100-year low flow is known to within roughly a factor of two, and a point estimate on its own invites over-reading.

Profile likelihood for the GEV, parametric bootstrap otherwise.

In [ ]:
rl = pd.DataFrame({"T": list(T)})
for nm, f in ok.items():
    rl[nm] = f.return_level(np.array(T, float))
print(rl.to_string(index=False))

In [ ]:
bf = ok[best]
ci = bf.return_level_ci(np.array(T, float), B=B_CI, rng=np.random.default_rng(1))
print(ci.to_string(index=False))

if "lower" in ci and (ci["lower"] < 0).any():
    print("\nWARNING: the interval extends below zero discharge at long return periods.")
    print("The fit is extrapolating past physical plausibility; prefer a lower-bounded")
    print("family or fit in log space before quoting those values.")
if isinstance(bf, GEVMinima):
    print(f"\nGEV shape = {bf.params['shape']:.4f}; implied lower bound on Q = {bf.lower_bound:.4g}")

In [ ]:
ci_band = bf.return_level_ci(np.logspace(np.log10(min(T)), np.log10(max(T)), 14),
                             B=B_CI, rng=np.random.default_rng(2))
fig = figs.fig_return_levels(ok, ci_band, x, best=best,
                             title=f"{STATION}: low-flow return levels", T=T)

## Write every table and figure to disk

The cells above are the analysis made explicit. This single call reruns it through the packaged pipeline and writes the full set of CSVs, PNG+PDF figures and a `run_log.txt` to `outdir` -- use it when you want the files rather than the narrative.

In [ ]:
from lowflow.pipeline import UnivariateConfig, run_univariate

cfg = UnivariateConfig(
    path=PATH, station=STATION, label=f'{STATION}, annual minimum {D}-day mean', outdir='out/nb01_marginal_annual',
    season=SEASON, D=D, min_coverage=MIN_COV, start_month=1,
    zero_policy="keep", T=T, B_gof=B_GOF, B_ci=B_CI,
    nonstationary=False, epochs=(),
    scan_season=True,
)
res = run_univariate(cfg)
print(f"\n{len(res['written'])} files written to {res['outdir']}")

Next: **02 — seasonal**, which asks whether the chosen low-flow window actually contains the drought.